In [7]:
!pip3 install google-adk -q
!pip3 install litellm -q
print("Installation complete")

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [6]:
!pip install google-adk[extensions] -q

zsh:1: no matches found: google-adk[extensions]


In [8]:
import os
import asyncio
from google.adk.agents import Agent 
from google.adk.models.lite_llm import LiteLlm 
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner 
from google.genai import types 

import warnings 
warnings.filterwarnings("ignore")

import logging 
logging.basicConfig(level = logging.ERROR) 
print("Libraries Imported") 


Libraries Imported


In [9]:
MODEL_GEMINI_2_5_FLASH = "gemini-flash-latest"


In [11]:
def get_weather(city: str)->dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city (e.g., "New York", "London", "Tokyo").

    Returns:
        dict: A dictionary containing the weather information.
              Includes a 'status' key ('success' or 'error').
              If 'success', includes a 'report' key with weather details.
              If 'error', includes an 'error_message' key.
    """
    print(f"--- Tool: get_weather called for city: {city} ---")
    city_normalized = city.lower().replace(" ","")

    mock_weather_db = {
        "newyork": {"status": "success", "report": "The weather in New York is sunny with a temperature of 25°C."},
        "london": {"status": "success", "report": "It's cloudy in London with a temperature of 15°C."},
        "tokyo": {"status": "success", "report": "Tokyo is experiencing light rain and a temperature of 18°C."},
    }

    if city_normalized in mock_weather_db: 
        return mock_weather_db[city_normalized]
    else: 
        return{"status":"error","error_message": f"Sorry, I don't have weather information for '{city}' ."}
print(get_weather("nEW yORK"))
print(get_weather("paris"))


--- Tool: get_weather called for city: nEW yORK ---
{'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25°C.'}
--- Tool: get_weather called for city: paris ---
{'status': 'error', 'error_message': "Sorry, I don't have weather information for 'paris' ."}


In [14]:
AGENT_MODEL = MODEL_GEMINI_2_5_FLASH 
weather_agent = Agent(
    name = "weather_agent_v1",
    model = AGENT_MODEL,
    description="Provides weather information for specific cities.",
    instruction="You are a helpful weather assistant. "
                "When the user asks for the weather in a specific city, "
                "use the 'get_weather' tool to find the information. "
                "If the tool returns an error, inform the user politely. "
                "If the tool is successful, present the weather report clearly.",
    tools =[get_weather],    
)
print(f"Agent '{weather_agent.name}' created using model '{AGENT_MODEL}'.")

Agent 'weather_agent_v1' created using model 'gemini-flash-latest'.


In [22]:
session_service =InMemorySessionService()
APP_NAME = "weather_tutorial_app"
USER_ID= "user_1"
SESSION_ID = "session_001"

session = await session_service.create_session(
    app_name = APP_NAME,
    user_id = USER_ID ,
    session_id = SESSION_ID

)
print(f"Session created : App ='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")
runner=Runner(agent=weather_agent,app_name=APP_NAME,session_service=session_service)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created : App ='weather_tutorial_app', User='user_1', Session='session_001'
Runner created for agent 'weather_agent_v1'.


In [23]:
from google.genai import types 
async def call_agent_async(query: str, runner , user_id, session_id):
    print(f"\n>>> User Query: {query}")

    content = types.Content(role='user',parts = [types.Part(text=query)])
    final_response_text= "Agent did not produce a final response."
    async for event in runner.run_async(user_id=user_id , session_id = session_id ,new_message= content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            elif event.actions and event.actions.escalate:
                final_response_text = f"Agent escalated: {event.error_message or 'No specific message'}"
            break 
    print(f"<<<Agent Response: {final_response_text}")

In [24]:
async def run_conversation():
    await call_agent_async("What is the weather like in London?",runner=runner,user_id=USER_ID,session_id=SESSION_ID)
    await call_agent_async("How about Paris?",runner=runner,user_id=USER_ID,session_id=SESSION_ID)
    await call_agent_async("Tell me the weather in new York",runner=runner, user_id=USER_ID,session_id=SESSION_ID)
await run_conversation()


>>> User Query: What is the weather like in London?
--- Tool: get_weather called for city: London ---


ERROR:opentelemetry.context:Failed to detach context
Traceback (most recent call last):
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/telemetry/_instrumentation.py", line 72, in record_invocation
    yield
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/runners.py", line 624, in _run_node_async
    yield event
GeneratorExit

During hand

<<<Agent Response: The weather in London is currently cloudy with a temperature of 15°C.

>>> User Query: How about Paris?
--- Tool: get_weather called for city: Paris ---


ERROR:opentelemetry.context:Failed to detach context
Traceback (most recent call last):
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/telemetry/_instrumentation.py", line 72, in record_invocation
    yield
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/runners.py", line 624, in _run_node_async
    yield event
GeneratorExit

During hand

<<<Agent Response: I'm sorry, but I don't have weather information for Paris at the moment.

>>> User Query: Tell me the weather in new York
--- Tool: get_weather called for city: New York ---
<<<Agent Response: The weather in New York is currently sunny with a temperature of 25°C.


ERROR:opentelemetry.context:Failed to detach context
Traceback (most recent call last):
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 608, in use_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 508, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/opentelemetry/trace/__init__.py", line 443, in start_as_current_span
    yield span
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/telemetry/_instrumentation.py", line 72, in record_invocation
    yield
  File "/Users/yatharthbisht/Desktop/Programming/exl-assignment/.venv/lib/python3.12/site-packages/google/adk/runners.py", line 624, in _run_node_async
    yield event
GeneratorExit

During hand